# Ejercicio 3: Análisis de Sentimientos en Reseñas de Productos

**Dataset:** Amazon Reviews Dataset  
**Objetivo:** Desarrollar un modelo de aprendizaje supervisado para clasificar reseñas de productos como positivas o negativas a partir de texto.

## Requerimientos

1. Limpieza y preprocesamiento del texto (stopwords, tokenización, TF-IDF)
2. Entrenar modelos de clasificación: Naive Bayes, Logistic Regression y SVM
3. Evaluar el rendimiento con Accuracy, F1-score y Matriz de Confusión
4. Reducción de dimensionalidad con t-SNE para visualizar agrupaciones de reseñas

In [20]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import ssl
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score
)
from sklearn.manifold import TSNE
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

for recurso in ['punkt_tab', 'stopwords', 'punkt']:
    nltk.download(recurso, quiet=True)

## 2. Carga del dataset

El dataset de Amazon Reviews está en formato FastText (`.ft.txt`), donde cada línea tiene la forma:


- `__label__1` = reseña negativa (1 o 2 estrellas)
- `__label__2` = reseña positiva (4 o 5 estrellas)

In [22]:
RUTA_DATASET = "../data/train.ft.txt"
TAMANO_MUESTRA = 200_000  

with open(RUTA_DATASET, 'r', encoding='utf-8') as archivo:
    lineas = archivo.readlines()

print(f"Total de registros en el archivo: {len(lineas):,}")
print(f"Registros que se usarán:          {TAMANO_MUESTRA:,}")

Total de registros en el archivo: 3,600,000
Registros que se usarán:          200,000


## 3. Parseo y construcción del DataFrame

Se extrae la etiqueta y el texto de cada línea, convirtiéndolas a valores binarios:
- 0 = Negativo (`__label__1`)
- 1 → Positivo (`__label__2`)

In [23]:
muestra = lineas[:TAMANO_MUESTRA]

datos = []
for linea in muestra:
    # Separar etiqueta y texto (solo en el primer espacio)
    partes = linea.strip().split(' ', 1)

    # Validar que la línea tenga el formato esperado
    if len(partes) != 2:
        continue

    label, texto = partes
    sentimiento = 0 if label == "__label__1" else 1  # 0=Negativo, 1=Positivo
    datos.append({"review": texto, "sentiment": sentimiento})

df = pd.DataFrame(datos)

print(f"Registros parseados: {len(df):,}")
print(f"Columnas:            {df.columns.tolist()}")
df.head()

Registros parseados: 200,000
Columnas:            ['review', 'sentiment']


,review,sentiment
0,Stuning even for the non-gamer: This sound tra...,1
1,The best soundtrack ever to anything.: I'm rea...,1
2,Amazing!: This soundtrack is my favorite music...,1
3,Excellent Soundtrack: I truly like this soundt...,1
4,"Remember, Pull Your Jaw Off The Floor After He...",1


In [24]:
# Distribución de clases
conteo = df["sentiment"].value_counts()
print("Distribución de clases:")
print(conteo.rename({0: 'Negativo (0)', 1: 'Positivo (1)'}).to_string())
print(f"\nBalanceo: {conteo[0]/len(df)*100:.1f}% neg  /  {conteo[1]/len(df)*100:.1f}% pos")

Distribución de clases:
sentiment
Positivo (1)    101166
Negativo (0)     98834

Balanceo: 49.4% neg  /  50.6% pos


## 4. Requerimiento 1 – Limpieza y preprocesamiento del texto

Se aplican los siguientes pasos de limpieza a cada reseña:

1. **Lowercase**: pasar todo a minúsculas para normalizar
2. **Eliminación de caracteres especiales**: conservar solo letras y espacios
3. **Tokenización**: dividir el texto en palabras individuales (`word_tokenize`)
4. **Eliminación de stopwords**: quitar palabras muy frecuentes sin valor semántico ("the", "is", "a", etc.)
5. **Reconstrucción**: unir los tokens filtrados en una cadena de texto

Posteriormente, se aplica **TF-IDF** para convertir el texto limpio en vectores numéricos que los modelos puedan procesar.

In [25]:
# Cargar stopwords en inglés
stop_words = set(stopwords.words('english'))


def limpiar_texto(texto: str) -> str:
    
    # Paso 1: minúsculas
    texto = texto.lower()

    # Paso 2: eliminar caracteres especiales (números, puntuación, etc.)
    texto = re.sub(r'[^a-z\s]', '', texto)

    # Paso 3: tokenización
    tokens = word_tokenize(texto)

    # Paso 4: eliminar stopwords y tokens de 1 carácter
    tokens = [t for t in tokens if t not in stop_words and len(t) > 1]

    # Paso 5: reconstruir texto
    return " ".join(tokens)


# Aplicar la función de limpieza a todas las reseñas
df["clean_review"] = df["review"].apply(limpiar_texto)

print("Ejemplo de preprocesamiento:")
print("ORIGINAL:", df["review"].iloc[0][:120])
print("LIMPIO:  ", df["clean_review"].iloc[0][:120])

Ejemplo de preprocesamiento:
ORIGINAL: Stuning even for the non-gamer: This sound track was beautiful! It paints the senery in your mind so well I would recome
LIMPIO:   stuning even nongamer sound track beautiful paints senery mind well would recomend even people hate vid game music playe


In [26]:
# TF-IDF 

vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),   
    sublinear_tf=True     
)

X = vectorizer.fit_transform(df["clean_review"])
y = df["sentiment"]

print(f"Matriz TF-IDF: {X.shape[0]:,} documentos × {X.shape[1]:,} características")

Matriz TF-IDF: 200,000 documentos × 5,000 características
